In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load CSV files into DataFrames and cache them in the kernel
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Ensure timestamp fields are numeric (integers) if possible
for df in (metric_df, trace_df, log_df, error_df):
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce").astype("Int64")

# Prepare metric aggregates
metric_total_rows = int(metric_df.shape[0])
metric_distinct_cmdb = int(metric_df["cmdb_id"].nunique()) if "cmdb_id" in metric_df.columns else 0
metric_top_cmdbs = metric_df["cmdb_id"].value_counts().head(20).reset_index()
metric_top_cmdbs.columns = ["cmdb_id", "count"]
metric_distinct_kpi = int(metric_df["kpi_name"].nunique()) if "kpi_name" in metric_df.columns else 0
metric_top_kpis = metric_df["kpi_name"].value_counts().head(20).reset_index()
metric_top_kpis.columns = ["kpi_name", "count"]
metric_ts_min = int(metric_df["timestamp"].min()) if "timestamp" in metric_df.columns and metric_df["timestamp"].notnull().any() else None
metric_ts_max = int(metric_df["timestamp"].max()) if "timestamp" in metric_df.columns and metric_df["timestamp"].notnull().any() else None

metric_summary = pd.DataFrame([{
    "total_rows": metric_total_rows,
    "distinct_cmdb": metric_distinct_cmdb,
    "distinct_kpi": metric_distinct_kpi,
    "timestamp_min": metric_ts_min,
    "timestamp_max": metric_ts_max
}])

# Prepare trace aggregates
trace_total_rows = int(trace_df.shape[0])
trace_distinct_cmdb = int(trace_df["cmdb_id"].nunique()) if "cmdb_id" in trace_df.columns else 0
trace_top_cmdbs = trace_df["cmdb_id"].value_counts().head(20).reset_index()
trace_top_cmdbs.columns = ["cmdb_id", "count"]
trace_distinct_trace = int(trace_df["trace_name"].nunique()) if "trace_name" in trace_df.columns else 0
trace_top_traces = trace_df["trace_name"].value_counts().head(20).reset_index()
trace_top_traces.columns = ["trace_name", "count"]
trace_ts_min = int(trace_df["timestamp"].min()) if "timestamp" in trace_df.columns and trace_df["timestamp"].notnull().any() else None
trace_ts_max = int(trace_df["timestamp"].max()) if "timestamp" in trace_df.columns and trace_df["timestamp"].notnull().any() else None

trace_summary = pd.DataFrame([{
    "total_rows": trace_total_rows,
    "distinct_cmdb": trace_distinct_cmdb,
    "distinct_trace_name": trace_distinct_trace,
    "timestamp_min": trace_ts_min,
    "timestamp_max": trace_ts_max
}])

# Prepare log aggregates
log_total_rows = int(log_df.shape[0])
log_distinct_cmdb = int(log_df["cmdb_id"].nunique()) if "cmdb_id" in log_df.columns else 0
log_top_cmdbs = log_df["cmdb_id"].value_counts().head(20).reset_index()
log_top_cmdbs.columns = ["cmdb_id", "count"]
log_distinct_logname = int(log_df["log_name"].nunique()) if "log_name" in log_df.columns else 0
log_top_logs = log_df["log_name"].value_counts().head(20).reset_index()
log_top_logs.columns = ["log_name", "count"]
log_ts_min = int(log_df["timestamp"].min()) if "timestamp" in log_df.columns and log_df["timestamp"].notnull().any() else None
log_ts_max = int(log_df["timestamp"].max()) if "timestamp" in log_df.columns and log_df["timestamp"].notnull().any() else None

log_summary = pd.DataFrame([{
    "total_rows": log_total_rows,
    "distinct_cmdb": log_distinct_cmdb,
    "distinct_log_name": log_distinct_logname,
    "timestamp_min": log_ts_min,
    "timestamp_max": log_ts_max
}])

# Prepare error_logs aggregates and example messages (up to 5 distinct)
error_total_rows = int(error_df.shape[0])
error_distinct_cmdb = int(error_df["cmdb_id"].nunique()) if "cmdb_id" in error_df.columns else 0
error_top_cmdbs = error_df["cmdb_id"].value_counts().head(20).reset_index()
error_top_cmdbs.columns = ["cmdb_id", "count"]
error_ts_min = int(error_df["timestamp"].min()) if "timestamp" in error_df.columns and error_df["timestamp"].notnull().any() else None
error_ts_max = int(error_df["timestamp"].max()) if "timestamp" in error_df.columns and error_df["timestamp"].notnull().any() else None

# Build up to 5 distinct example messages with earliest timestamp and example cmdb_id
if "message" in error_df.columns:
    # Normalize message to string and truncate for display
    error_df["message_str"] = error_df["message"].astype(str)
    grouped = (
        error_df.sort_values("timestamp", na_position="last")
        .groupby("message_str", sort=False, as_index=False)
        .agg(example_timestamp=("timestamp", "first"),
             example_cmdb_id=("cmdb_id", lambda x: x.iloc[0]),
             occurrences=("message_str", "count"))
    )
    # Truncate messages for compact display
    grouped["message_short"] = grouped["message_str"].str.slice(0, 120)
    error_examples = grouped.sort_values(["example_timestamp", "occurrences"]).head(5)[
        ["example_timestamp", "example_cmdb_id", "occurrences", "message_short"]
    ].reset_index(drop=True)
else:
    error_examples = pd.DataFrame(columns=["example_timestamp", "example_cmdb_id", "occurrences", "message_short"])

error_summary = pd.DataFrame([{
    "total_rows": error_total_rows,
    "distinct_cmdb": error_distinct_cmdb,
    "timestamp_min": error_ts_min,
    "timestamp_max": error_ts_max
}])

# Final compact outputs (saved DataFrames are metric_df, trace_df, log_df, error_df)
metric_summary, metric_top_cmdbs.head(20), metric_top_kpis.head(20), trace_summary, trace_top_cmdbs.head(20), trace_top_traces.head(20), log_summary, log_top_cmdbs.head(20), log_top_logs.head(20), error_summary, error_top_cmdbs.head(20), error_examples

```
Out[1]:
```
summary = (
    "Summary of loaded telemetry files:\n"
    "1) metric.csv: 149,652 rows; 59 distinct cmdb_id. Top cmdb_id by row count (top 5): "
    "adservice (10230), adservice2 (9990), frontend2-0 (5280), frontend-0 (5160), frontend-1 (5160). "
    "1244 distinct kpi_name (top kpis dominated by container.node-5.* series). "
    "timestamp range: 1647840600 – 1647842340.\n\n"
    "2) trace.csv: 33,320 rows; 40 distinct cmdb_id. Top cmdb_id (top 5): "
    "frontend-1 (2752), frontend-0 (2700), checkoutservice-2 (2428), checkoutservice-0 (2424), checkoutservice-1 (2404). "
    "200 distinct trace_name. timestamp range: 1647840600 – 1647842340.\n\n"
    "3) log.csv: 1,804 rows; 31 distinct cmdb_id. Many cmdbs have 60 rows each (e.g., adservice-*, cartservice-*, checkoutservice-*). "
    "2 distinct log_name: log.error_count (902) and log.row_count (902). "
    "timestamp range: 1647840600 – 1647842340.\n\n"
    "4) error_logs.csv: 1,093 rows; 3 distinct cmdb_id. Top cmdb_id: frontend-1 (574), frontend-0 (414), frontend-2 (105). "
    "Example messages (up to 5): the primary message 'severity: warning, message: failed to retrieve...' at timestamp 1647840601 from frontend-2 (occurring ~1090 times), "
    "and 'severity: error, message: request error' at 1647841823 from frontend-2 (occurring ~3 times). "
    "timestamp range: 1647840601 – 1647842398.\n\n"
    "All outputs are cached in metric_df, trace_df, log_df, error_df for further analysis."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(   total_rows  distinct_cmdb  distinct_kpi  timestamp_min  timestamp_max
0      149652             59          1244     1647840600     1647842340,                      cmdb_id  count
0                  adservice  10230
1                 adservice2   9990
2                frontend2-0   5280
3                 frontend-0   5160
4                 frontend-1   5160
5                 frontend-2   4980
6          checkoutservice-0   4020
7          checkoutservice-2   4020
8          checkoutservice-1   3780
9         checkoutservice2-0   3540
10   productcatalogservice-1   3480
11   productcatalogservice-2   3480
12   productcatalogservice-0   3480
13              adservice2-0   3060
14             cartservice-1   3000
15  productcatalogservice2-0   2880
16             cartservice-0   2880
17            cartservice2-0   2880
18         shippingservice-0   2640
19             cartservice-2   2640,                                              kpi_name  count
0   container.node-5.container_network_transmit_er...   1050
1   container.node-5.container_network_transmit_pa...   1050
2   container.node-5.container_network_receive_err...   1050
3   container.node-5.container_network_transmit_pa...   1050
4   container.node-5.container_network_receive_MB....   1050
5   container.node-5.container_network_receive_pac...   1050
6   container.node-5.container_network_transmit_MB...   1050
7   container.node-5.container_network_receive_pac...   1050
8    container.node-5.container_tasks_state.iowaiting    990
9      container.node-5.container_tasks_state.stopped    990
10    container.node-5.container_tasks_state.sleeping    990
11     container.node-5.container_tasks_state.running    990
12  container.node-5.container_fs_io_time_seconds....    990
13         container.node-5.container_cpu_cfs_periods    990
14  container.node-5.container_fs_io_time_weighted...    990
15   container.node-5.container_fs_limit_MB./dev/vda1    990
16  container.node-5.container_fs_read_seconds./de...    990
17      container.node-5.container_fs_reads./dev/vda1    990
18  container.node-5.container_cpu_cfs_throttled_p...    990
19  container.node-5.container_cpu_cfs_throttled_s...    990,    total_rows  distinct_cmdb  distinct_trace_name  timestamp_min  timestamp_max
0       33320             40                  200     1647840600     1647842340,                     cmdb_id  count
0                frontend-1   2752
1                frontend-0   2700
2         checkoutservice-2   2428
3         checkoutservice-0   2424
4         checkoutservice-1   2404
5                frontend-2   1648
6   productcatalogservice-2   1152
7   productcatalogservice-0   1144
8   productcatalogservice-1   1136
9               frontend2-0   1080
10       checkoutservice2-0   1080
11  recommendationservice-0    800
12  recommendationservice-2    800
13  recommendationservice-1    800
14            cartservice-0    792
15            cartservice-1    792
16            cartservice-2    780
17        currencyservice-1    680
18        currencyservice-2    676
19        currencyservice-0    676,                                     trace_name  count
0              trace.from_frontend-1.row_count    628
1             trace.from_frontend-1.error_rate    628
2           trace.from_frontend-1.duration_p95    628
3          trace.from_frontend-1.duration_mean    628
4              trace.from_frontend-0.row_count    615
5             trace.from_frontend-0.error_rate    615
6           trace.from_frontend-0.duration_p95    615
7          trace.from_frontend-0.duration_mean    615
8       trace.from_checkoutservice-2.row_count    487
9      trace.from_checkoutservice-2.error_rate    487
10   trace.from_checkoutservice-2.duration_p95    487
11  trace.from_checkoutservice-2.duration_mean    487
12      trace.from_checkoutservice-0.row_count    484
13     trace.from_checkoutservice-0.error_rate    484
14   trace.from_checkoutservice-0.duration_p95    484
15  trace.from_checkoutservice-0.duration_mean    484
16      trace.from_checkoutservice-1.row_count    480
17     trace.from_checkoutservice-1.error_rate    480
18   trace.from_checkoutservice-1.duration_p95    480
19  trace.from_checkoutservice-1.duration_mean    480,    total_rows  distinct_cmdb  distinct_log_name  timestamp_min  timestamp_max
0        1804             31                  2     1647840600     1647842340,                     cmdb_id  count
0               adservice-0     60
1               adservice-1     60
2               adservice-2     60
3             cartservice-0     60
4             cartservice-1     60
5             cartservice-2     60
6         checkoutservice-0     60
7         checkoutservice-1     60
8         checkoutservice-2     60
9         currencyservice-0     60
10        currencyservice-1     60
11        currencyservice-2     60
12           emailservice-0     60
13           emailservice-1     60
14               frontend-0     60
15        shippingservice-0     60
16               frontend-1     60
17         paymentservice-0     60
18         paymentservice-2     60
19  productcatalogservice-1     60,           log_name  count
0  log.error_count    902
1    log.row_count    902,    total_rows  distinct_cmdb  timestamp_min  timestamp_max
0        1093              3     1647840601     1647842398,       cmdb_id  count
0  frontend-1    574
1  frontend-0    414
2  frontend-2    105,    example_timestamp example_cmdb_id  occurrences                                      message_short
0         1647840601      frontend-2         1090  severity: warning, message: failed to retrieve...
1         1647841823      frontend-2            3            severity: error, message: request error)```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# Reuse existing metric_df loaded earlier in the kernel.

# 1) Compute global P95 per (cmdb_id, kpi_name) using full series
thresholds_df = (
    metric_df
    .groupby(["cmdb_id", "kpi_name"], as_index=False)["value"]
    .quantile(0.95)
    .rename(columns={"value": "p95"})
)

# 2) Filter metric_df to the incident time window Mar 21, 2022 13:30:00 to 14:00:00 in UTC+8
tz = pytz.timezone("Asia/Shanghai")
start_ts = int(pd.Timestamp("2022-03-21 13:30:00", tz=tz).timestamp())
end_ts = int(pd.Timestamp("2022-03-21 14:00:00", tz=tz).timestamp())

metric_incident_df = metric_df[(metric_df["timestamp"] >= start_ts) & (metric_df["timestamp"] <= end_ts)].copy()

# 3) For each pair present in the incident window compute required metrics
# Join p95 thresholds to the incident data
metric_incident_with_thresh = metric_incident_df.merge(thresholds_df, on=["cmdb_id", "kpi_name"], how="left")

# Flag anomalies where value >= global p95
metric_incident_with_thresh["is_anomaly"] = metric_incident_with_thresh["value"] >= metric_incident_with_thresh["p95"]

# Aggregations
counts = (
    metric_incident_with_thresh.groupby(["cmdb_id", "kpi_name"], as_index=False)
    .size()
    .rename(columns={"size": "total_points_in_window"})
)

anomaly_counts = (
    metric_incident_with_thresh.groupby(["cmdb_id", "kpi_name"], as_index=False)["is_anomaly"]
    .sum()
    .rename(columns={"is_anomaly": "anomaly_count_in_window"})
)

earliest_anom = (
    metric_incident_with_thresh[metric_incident_with_thresh["is_anomaly"]]
    .groupby(["cmdb_id", "kpi_name"], as_index=False)["timestamp"]
    .min()
    .rename(columns={"timestamp": "earliest_anomaly_timestamp_in_window"})
)

max_vals = (
    metric_incident_with_thresh.groupby(["cmdb_id", "kpi_name"], as_index=False)["value"]
    .max()
    .rename(columns={"value": "max_value_in_window"})
)

# Merge aggregated results and join p95
summary = counts.merge(anomaly_counts, on=["cmdb_id", "kpi_name"], how="left")
summary = summary.merge(earliest_anom, on=["cmdb_id", "kpi_name"], how="left")
summary = summary.merge(max_vals, on=["cmdb_id", "kpi_name"], how="left")
summary = summary.merge(thresholds_df, on=["cmdb_id", "kpi_name"], how="left")

# Keep column order requested and ensure compactness
summary_table = summary[
    [
        "cmdb_id",
        "kpi_name",
        "total_points_in_window",
        "anomaly_count_in_window",
        "earliest_anomaly_timestamp_in_window",
        "p95",
        "max_value_in_window",
    ]
]

# Fill NaNs for anomaly counts with 0 (if any) and cast earliest timestamp to Int64 for compactness
summary_table["anomaly_count_in_window"] = summary_table["anomaly_count_in_window"].fillna(0).astype(int)
summary_table["earliest_anomaly_timestamp_in_window"] = summary_table["earliest_anomaly_timestamp_in_window"].astype("Int64")

# 4) Return top 20 by anomaly_count_in_window (desc)
summary_table = summary_table.sort_values(by=["anomaly_count_in_window", "max_value_in_window"], ascending=[False, False]).reset_index(drop=True)

# Expose thresholds_df and metric_incident_df in kernel for next steps (they already exist as variables)
thresholds_df, metric_incident_df.shape, summary_table.head(20)

```
Out[2]:
```
summary = (
    "Results of the analysis (compact):\n"
    "- thresholds_df: 4,998 unique (cmdb_id, kpi_name) pairs with global P95 computed.\n"
    "- metric_incident_df: 149,652 rows (filtered window Mar 21, 2022 13:30–14:00 UTC+8) saved for next steps.\n"
    "- Per-(cmdb_id,kpi_name) anomaly summary (top 20 by anomaly count): many entries have anomaly_count_in_window = 30 (i.e., every point in the window >= the global P95).\n"
    "- The top anomalous KPIs are largely static/monotonic metrics such as runtime.java_lang_Runtime_StartTime, container_start_time_seconds, and system.disk.total (node-level). Their P95 values are very large (e.g., start_time ≈ 1.647e+12, disk totals ≈ 3.6–7.6e+9) and max_value_in_window equals those P95s, with earliest anomaly timestamps at 1647840600.\n"
    "- Interpretation: the highest-ranked 'anomalies' are dominated by metrics that naturally hold large or monotonic values (start times, total disk size), so they may be false-positives for behaviour anomalies. For incident triage, prioritize dynamic operational KPIs (CPU, memory, network, latency) rather than these static metrics.\n"
    "- Saved variables for next steps: thresholds_df and metric_incident_df."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                                           kpi_name            p95
0              adservice                                     app.grpc.count     252.650000
1              adservice                                       app.grpc.mrt       0.002434
2              adservice                                        app.grpc.rr     100.000000
3              adservice                                        app.grpc.sr      86.053239
4              adservice                                     app.http.count      24.000000
...                  ...                                                ...            ...
4993  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...      48.550000
4994  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....  119752.500000
4995  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....       0.002064
4996  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....      32.775000
4997  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....    3320.812500

[4998 rows x 3 columns], (149652, 4),                      cmdb_id                                           kpi_name  total_points_in_window  anomaly_count_in_window  earliest_anomaly_timestamp_in_window           p95  max_value_in_window
0                 adservice2                runtime.java_lang_Runtime_StartTime                      30                       30                            1647840600  1.647822e+12         1.647822e+12
1                  adservice                runtime.java_lang_Runtime_StartTime                      30                       30                            1647840600  1.647523e+12         1.647523e+12
2                     node-4                                  system.disk.total                      30                       30                            1647840600  7.616300e+09         7.616300e+09
3                     node-1                                  system.disk.total                      30                       30                            1647840600  5.867011e+09         5.867011e+09
4                     node-2                                  system.disk.total                      30                       30                            1647840600  5.867011e+09         5.867011e+09
5                     node-3                                  system.disk.total                      30                       30                            1647840600  5.867011e+09         5.867011e+09
6                     node-6                                  system.disk.total                      30                       30                            1647840600  3.617924e+09         3.617924e+09
7                  adservice  runtime.java_lang_OperatingSystem_CommittedVir...                      30                       30                            1647840600  3.589956e+09         3.589956e+09
8                 adservice2  runtime.java_lang_OperatingSystem_CommittedVir...                      30                       30                            1647840600  3.430343e+09         3.430343e+09
9         currencyservice2-0          container.node-6.container_memory_failcnt                      30                       30                            1647840600  1.659063e+09         1.659063e+09
10              adservice2-0      container.node-6.container_start_time_seconds                      30                       30                            1647840600  1.647822e+09         1.647822e+09
11             cartservice-0      container.node-5.container_start_time_seconds                      30                       30                            1647840600  1.647778e+09         1.647778e+09
12             cartservice-1      container.node-5.container_start_time_seconds                      30                       30                            1647840600  1.647778e+09         1.647778e+09
13             cartservice-2      container.node-5.container_start_time_seconds                      30                       30                            1647840600  1.647778e+09         1.647778e+09
14  productcatalogservice2-0      container.node-6.container_start_time_seconds                      30                       30                            1647840600  1.647523e+09         1.647523e+09
15        currencyservice2-0      container.node-6.container_start_time_seconds                      30                       30                            1647840600  1.647523e+09         1.647523e+09
16         paymentservice2-0      container.node-6.container_start_time_seconds                      30                       30                            1647840600  1.647523e+09         1.647523e+09
17  recommendationservice2-0      container.node-6.container_start_time_seconds                      30                       30                            1647840600  1.647523e+09         1.647523e+09
18               frontend2-0      container.node-6.container_start_time_seconds                      30                       30                            1647840600  1.647523e+09         1.647523e+09
19   productcatalogservice-2      container.node-5.container_start_time_seconds                      30                       30                            1647840600  1.647523e+09         1.647523e+09)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import re

# Reuse thresholds_df and metric_incident_df from kernel

# 1) Build case-insensitive regex pattern from substrings
substrs = [
    'cpu', 'memory', 'network', 'transmit', 'receive', 'packet', 'latency',
    'duration', 'error_rate', 'iowait', 'iow', 'io', 'read', 'write', 'fail',
    'throttl', 'disk.used', 'disk.io', 'fs_read', 'fs_write'
]
# Escape dot in substrings where necessary and join into regex
escaped = [re.escape(s) for s in substrs]
pattern = "(" + "|".join(escaped) + ")"

# Ensure kpi_name is string
thresholds_df["kpi_name"] = thresholds_df["kpi_name"].astype(str)

# Filter thresholds for matching kpi_name
candidate_thresholds = thresholds_df[thresholds_df["kpi_name"].str.contains(pattern, case=False, regex=True)].copy()

# 2) Compute counts in incident window per (cmdb_id, kpi_name)
counts_incident = (
    metric_incident_df.groupby(["cmdb_id", "kpi_name"], as_index=False)
    .size()
    .rename(columns={"size": "total_points_in_window"})
)

# 3) Join counts with candidate thresholds; keep only pairs present in the incident window
candidates_metrics_df = counts_incident.merge(
    candidate_thresholds,
    on=["cmdb_id", "kpi_name"],
    how="inner"
)[["cmdb_id", "kpi_name", "total_points_in_window", "p95"]]

# 4) Sort by count desc, then kpi_name asc for ties; limit to top 50
candidates_metrics_df = candidates_metrics_df.sort_values(
    by=["total_points_in_window", "kpi_name"],
    ascending=[False, True]
).reset_index(drop=True)

candidates_metrics_df = candidates_metrics_df.head(50)

# Display the compact result (stored in kernel as candidates_metrics_df)
candidates_metrics_df

```
Out[3]:
```
summary = (
    f"Dynamic operational KPI candidates (from thresholds_df ∩ incident window):\n"
    f"- Total candidate (cmdb_id, kpi_name) pairs returned: {candidates_metrics_df.shape[0]}.\n"
    f"- Unique kpi_name count: {candidates_metrics_df['kpi_name'].nunique()}; unique cmdb_id count: {candidates_metrics_df['cmdb_id'].nunique()}.\n"
    f"- Most pairs have full coverage in the incident window: {round((candidates_metrics_df['total_points_in_window']==30).mean()*100,1)}% have 30 points.\n"
    f"- Top observed KPI examples (by frequency):\n"
    f"  1) container.node-5.container_cpu_cfs_periods (appears across many containers) — CPU scheduling metric.\n"
    f"  2) container.node-5.container_cpu_cfs_throttled_* (many entries with p95 = 0) — CPU throttling indicators.\n"
    f"  3) various container memory and IO/network related KPI names also matched the filter.\n"
    f"- Interpretation & recommendation: these candidates are dominated by CPU-related metrics (CFS periods and throttling) tied to node-5 and many containers. Because many entries have full 30-point coverage, this indicates sustained high/consistent values during the incident window rather than sporadic spikes. Investigate node-5 CPU contention and container CPU throttling (cfs throttling, cfs_periods) first, and then check memory, network, and IO KPIs among the listed candidates.\n"
    f"- The filtered candidates table is saved as candidates_metrics_df for next steps."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  total_points_in_window      p95
0         adservice-0         container.node-5.container_cpu_cfs_periods                      30  121.825
1         adservice-1         container.node-5.container_cpu_cfs_periods                      30   98.325
2         adservice-2         container.node-5.container_cpu_cfs_periods                      30   96.100
3       cartservice-0         container.node-5.container_cpu_cfs_periods                      30  201.275
4       cartservice-1         container.node-5.container_cpu_cfs_periods                      30  210.275
..                ...                                                ...                     ...      ...
45  currencyservice-2  container.node-5.container_cpu_cfs_throttled_p...                      30    0.000
46     emailservice-0  container.node-5.container_cpu_cfs_throttled_p...                      30    0.000
47     emailservice-1  container.node-5.container_cpu_cfs_throttled_p...                      30    0.000
48     emailservice-2  container.node-5.container_cpu_cfs_throttled_p...                      30    0.000
49    emailservice2-0  container.node-5.container_cpu_cfs_throttled_p...                      30    0.000

[50 rows x 4 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse cached variables: candidates_metrics_df, metric_incident_df, thresholds_df

# 1) Filter incident-window rows to only candidate (cmdb_id, kpi_name) pairs and join p95
candidates_pairs = candidates_metrics_df[["cmdb_id", "kpi_name"]].drop_duplicates()
candidates_incident = metric_incident_df.merge(candidates_pairs, on=["cmdb_id", "kpi_name"], how="inner")
candidates_incident = candidates_incident.merge(thresholds_df, on=["cmdb_id", "kpi_name"], how="left")

# Ensure p95 present; mark anomalies (value >= p95). If p95 is NaN, treat as not anomaly.
candidates_incident["is_anomaly"] = False
mask_has_p95 = candidates_incident["p95"].notna()
candidates_incident.loc[mask_has_p95, "is_anomaly"] = (
    candidates_incident.loc[mask_has_p95, "value"] >= candidates_incident.loc[mask_has_p95, "p95"]
)

# 2) Extract only anomaly rows and group contiguous (minute-step) timestamps into segments
anomalies = candidates_incident[candidates_incident["is_anomaly"]].copy()

if not anomalies.empty:
    # Sort to ensure ordering
    anomalies = anomalies.sort_values(["cmdb_id", "kpi_name", "timestamp"]).reset_index(drop=True)

    # compute timestamp diff within each (cmdb_id, kpi_name)
    anomalies["ts_diff"] = anomalies.groupby(["cmdb_id", "kpi_name"])["timestamp"].diff().fillna(0).astype("Int64")

    # start a new segment when the gap is not 60 seconds (not contiguous minute)
    anomalies["new_segment_flag"] = (anomalies["ts_diff"] != 60).astype(int)

    # cumulative sum of new_segment_flag within each group gives a segment index per pair
    anomalies["segment_idx"] = anomalies.groupby(["cmdb_id", "kpi_name"])["new_segment_flag"].cumsum()

    # 3) Aggregate per segment
    faults_df = (
        anomalies
        .groupby(["cmdb_id", "kpi_name", "segment_idx"], as_index=False)
        .agg(
            fault_start_timestamp=("timestamp", "min"),
            fault_end_timestamp=("timestamp", "max"),
            length_in_points=("timestamp", "count"),
            peak_value_in_segment=("value", "max")
        )
    )

    # attach p95 for each pair
    faults_df = faults_df.merge(thresholds_df, on=["cmdb_id", "kpi_name"], how="left")

    # cleanup types for compactness
    faults_df["fault_start_timestamp"] = faults_df["fault_start_timestamp"].astype("Int64")
    faults_df["fault_end_timestamp"] = faults_df["fault_end_timestamp"].astype("Int64")
else:
    # No anomalies found -> empty faults_df with expected columns
    faults_df = pd.DataFrame(
        columns=[
            "cmdb_id", "kpi_name", "segment_idx", "fault_start_timestamp",
            "fault_end_timestamp", "length_in_points", "peak_value_in_segment", "p95"
        ]
    )

# 4) Prepare compact top-20 table sorted by length_in_points desc, tie-breaker by kpi_name
faults_top20 = (
    faults_df
    .sort_values(by=["length_in_points", "kpi_name"], ascending=[False, True])
    .reset_index(drop=True)
    .loc[:, ["cmdb_id", "kpi_name", "fault_start_timestamp", "fault_end_timestamp", "length_in_points", "peak_value_in_segment", "p95"]]
    .head(20)
)

# Expose faults_df and return compact result (shape for full faults_df and the top-20 segments)
faults_df.shape, faults_top20

```
Out[4]:
```
summary = (
    "Fault detection summary (compact):\n"
    "- faults_df contains 99 detected fault segments in the incident window.\n"
    "- The longest segments (top hits) are mostly container CPU throttling metrics (container.node-5.container_cpu_cfs_throttled_...) "
    "appearing across many containers (e.g., adservice-*, cartservice-*, emailservice-*, currencyservice-*). Several of these segments span the entire window "
    "(fault_start 1647840600 to fault_end 1647842340) with length_in_points = 30.\n"
    "- Many of those long throttling segments have p95 = 0 and peak_value_in_segment = 0.0. Because the global p95 is zero, the anomaly rule (value >= p95) "
    "marks all points as anomalies — this can create false-positives and explains the full-window segments.\n"
    "- In addition to full-window throttling hits, there are shorter, real-looking events for container_cpu_cfs_periods (single-minute spikes) and a few 2-point segments "
    "for throttling on some services.\n"
    "- Recommendations: prioritize investigation of node-5 CPU behavior and container CPU throttling (cfs throttling and cfs_periods) but first validate thresholds: "
    "treat p95==0 cases with caution (adjust thresholding or use a secondary rule) so static/zero-baseline metrics don't dominate the alarm list.\n"
    "- Variables saved for follow-up: faults_df, candidates_metrics_df, metric_incident_df, thresholds_df."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

((99, 8),               cmdb_id                                           kpi_name  fault_start_timestamp  fault_end_timestamp  length_in_points  peak_value_in_segment      p95
0         adservice-0  container.node-5.container_cpu_cfs_throttled_p...             1647840600           1647842340                30                    0.0    0.000
1         adservice-1  container.node-5.container_cpu_cfs_throttled_p...             1647840600           1647842340                30                    0.0    0.000
2         adservice-2  container.node-5.container_cpu_cfs_throttled_p...             1647840600           1647842340                30                    0.0    0.000
3       cartservice-0  container.node-5.container_cpu_cfs_throttled_p...             1647840600           1647842340                30                    2.0    0.000
4       cartservice-2  container.node-5.container_cpu_cfs_throttled_p...             1647840600           1647842340                30                    1.0    0.000
5   currencyservice-1  container.node-5.container_cpu_cfs_throttled_p...             1647840600           1647842340                30                    0.0    0.000
6   currencyservice-2  container.node-5.container_cpu_cfs_throttled_p...             1647840600           1647842340                30                    0.0    0.000
7      emailservice-0  container.node-5.container_cpu_cfs_throttled_p...             1647840600           1647842340                30                    0.0    0.000
8      emailservice-1  container.node-5.container_cpu_cfs_throttled_p...             1647840600           1647842340                30                    0.0    0.000
9      emailservice-2  container.node-5.container_cpu_cfs_throttled_p...             1647840600           1647842340                30                    0.0    0.000
10    emailservice2-0  container.node-5.container_cpu_cfs_throttled_p...             1647840600           1647842340                30                    0.0    0.000
11      cartservice-1  container.node-5.container_cpu_cfs_throttled_p...             1647841020           1647841080                 2                    1.0    0.550
12  checkoutservice-2  container.node-5.container_cpu_cfs_throttled_p...             1647841320           1647841380                 2                    2.0    2.000
13  currencyservice-0  container.node-5.container_cpu_cfs_throttled_p...             1647840840           1647840900                 2                    0.5    0.500
14  currencyservice-0  container.node-5.container_cpu_cfs_throttled_p...             1647842280           1647842340                 2                    0.5    0.500
15        adservice-0         container.node-5.container_cpu_cfs_periods             1647841020           1647841020                 1                  122.5  121.825
16        adservice-0         container.node-5.container_cpu_cfs_periods             1647841260           1647841260                 1                  125.0  121.825
17        adservice-1         container.node-5.container_cpu_cfs_periods             1647841560           1647841560                 1                  110.5   98.325
18        adservice-1         container.node-5.container_cpu_cfs_periods             1647842280           1647842280                 1                   99.0   98.325
19        adservice-2         container.node-5.container_cpu_cfs_periods             1647841140           1647841140                 1                   98.0   96.100)```
```

In [ ]:
# In[5]:

import numpy as np
import pandas as pd

# Reuse faults_df and thresholds_df from kernel

# Make a working copy
refined_faults_df = faults_df.copy()

# 1) Compute breach_ratio = (peak_value_in_segment - p95) / max(abs(p95), 1e-6)
# Ensure numeric values
refined_faults_df["p95"] = pd.to_numeric(refined_faults_df["p95"], errors="coerce")
refined_faults_df["peak_value_in_segment"] = pd.to_numeric(refined_faults_df["peak_value_in_segment"], errors="coerce")

denom = np.maximum(refined_faults_df["p95"].abs(), 1e-6)
refined_faults_df["breach_ratio"] = (refined_faults_df["peak_value_in_segment"] - refined_faults_df["p95"]) / denom

# 2) Filter: breach_ratio >= 0.5 AND length_in_points >= 2
refined_faults_df = refined_faults_df[
    (refined_faults_df["breach_ratio"] >= 0.5) &
    (refined_faults_df["length_in_points"] >= 2)
].copy()

# 3) Save refined_faults_df in kernel (already assigned)

# 4a) Top 20 rows sorted by length_in_points (desc), then breach_ratio (desc)
refined_top20 = (
    refined_faults_df
    .sort_values(by=["length_in_points", "breach_ratio"], ascending=[False, False])
    .reset_index(drop=True)
    .loc[:, ["cmdb_id", "kpi_name", "fault_start_timestamp", "fault_end_timestamp",
             "length_in_points", "peak_value_in_segment", "p95", "breach_ratio"]]
)

# Round breach_ratio and peak/p95 for compact display
refined_top20["breach_ratio"] = refined_top20["breach_ratio"].round(4)
refined_top20["peak_value_in_segment"] = refined_top20["peak_value_in_segment"].round(6)
refined_top20["p95"] = refined_top20["p95"].round(6)

refined_top20 = refined_top20.head(20)

# 4b) Aggregate counts of refined faults by fault_start_timestamp (top 20 by count)
start_counts = (
    refined_faults_df
    .groupby("fault_start_timestamp", as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .sort_values(by=["count", "fault_start_timestamp"], ascending=[False, True])
    .head(20)
)

# Ensure timestamp column is compact integer type
start_counts["fault_start_timestamp"] = start_counts["fault_start_timestamp"].astype("Int64")

# Return the two compact outputs
refined_top20, start_counts

```
Out[5]:
```
summary = (
    "Refined faults summary (compact):\n"
    "- Detected high-confidence faults after filtering (breach_ratio>=0.5 and length>=2): 3 segments.\n"
    "- Top segments:\n"
    "  1) cartservice-0, container.node-5.container_cpu_cfs_throttled_..., start=1647840600 end=1647842340, length=30, peak=2.0, p95=0.00, breach_ratio very large (p95==0 -> inflated).\n"
    "  2) cartservice-2, container.node-5.container_cpu_cfs_throttled_..., start=1647840600 end=1647842340, length=30, peak=1.0, p95=0.00, breach_ratio very large (p95==0 -> inflated).\n"
    "  3) cartservice-1, container.node-5.container_cpu_cfs_throttled_..., start=1647841020 end=1647841080, length=2, peak=1.0, p95=0.55, breach_ratio≈0.8182.\n"
    "- Fault start counts (top): timestamp 1647840600 -> 2 faults; 1647841020 -> 1 fault.\n"
    "- Interpretation & action: these refined faults point to container CPU throttling on node-5 affecting cartservice pods. However, many p95 values are zero which inflates breach_ratio — validate thresholds (avoid p95==0 as sole rule) and then investigate node-5 CPU contention and container throttling metrics for the listed cartservice pods."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id                                           kpi_name  fault_start_timestamp  fault_end_timestamp  length_in_points  peak_value_in_segment   p95  breach_ratio
0  cartservice-0  container.node-5.container_cpu_cfs_throttled_p...             1647840600           1647842340                30                    2.0  0.00  2.000000e+06
1  cartservice-2  container.node-5.container_cpu_cfs_throttled_p...             1647840600           1647842340                30                    1.0  0.00  1.000000e+06
2  cartservice-1  container.node-5.container_cpu_cfs_throttled_p...             1647841020           1647841080                 2                    1.0  0.55  8.182000e-01,    fault_start_timestamp  count
0             1647840600      2
1             1647841020      1)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import re
import pytz

# Reuse cached trace_df and refined_faults_df

# 1) Compute global P95 and P5 per (cmdb_id, trace_name)
trace_thresholds_df = (
    trace_df
    .groupby(["cmdb_id", "trace_name"], as_index=False)["value"]
    .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
)

# 2) Prepare list of distinct fault_start_timestamp from refined_faults_df
start_timestamps = sorted(refined_faults_df["fault_start_timestamp"].dropna().unique().tolist())

# build set of timestamps to extract: each start and start+60
timestamps_to_check = sorted({ts for s in start_timestamps for ts in (int(s), int(s) + 60)})

# Extract trace rows at those timestamps
trace_checks = trace_df[trace_df["timestamp"].isin(timestamps_to_check)].copy()
# attach which fault_start_timestamp they correspond to (map t or t-60)
# For a checked timestamp t_checked:
#  - if t_checked equals a fault_start_timestamp -> assign that start
#  - if t_checked equals fault_start_timestamp + 60 -> assign that start (t_checked-60)
def map_to_start(ts):
    if ts in start_timestamps:
        return int(ts)
    if (ts - 60) in start_timestamps:
        return int(ts - 60)
    return pd.NA

trace_checks["fault_start_timestamp"] = trace_checks["timestamp"].apply(map_to_start).astype("Int64")

# 2b) Join thresholds (p95 and p5)
trace_checks = trace_checks.merge(trace_thresholds_df, on=["cmdb_id", "trace_name"], how="left")

# 3) Classify anomaly type and flag anomalies
low_indicators = ["row_count", "count", "throughput", "error_rate", "error", "rate"]
high_indicators = ["duration", "latency", "mean", "p95", "p99", "duration_p95", "duration_mean"]

def classify(name):
    n = name.lower()
    if any(tok in n for tok in low_indicators):
        return "low"
    if any(tok in n for tok in high_indicators):
        return "high"
    # default heuristic: treat 'duration' like high, else high
    return "high"

trace_checks["anomaly_type_guess"] = trace_checks["trace_name"].astype(str).apply(classify)

# Ensure numeric thresholds
trace_checks["p95"] = pd.to_numeric(trace_checks["p95"], errors="coerce")
trace_checks["p5"] = pd.to_numeric(trace_checks["p5"], errors="coerce")
trace_checks["value"] = pd.to_numeric(trace_checks["value"], errors="coerce")

eps = 1e-6
# compute deviations and flags
def compute_row_flags(df):
    # high anomalies: value >= p95
    df["is_high_anomaly"] = False
    mask_high = df["anomaly_type_guess"] == "high"
    mask_high_has = mask_high & df["p95"].notnull()
    denom_high = np.maximum(df.loc[mask_high_has, "p95"].abs(), eps)
    df.loc[mask_high_has, "deviation"] = (df.loc[mask_high_has, "value"] - df.loc[mask_high_has, "p95"]) / denom_high
    df.loc[mask_high_has, "is_high_anomaly"] = df.loc[mask_high_has, "value"] >= df.loc[mask_high_has, "p95"]

    # low anomalies: value <= p5
    df["is_low_anomaly"] = False
    mask_low = df["anomaly_type_guess"] == "low"
    mask_low_has = mask_low & df["p5"].notnull()
    denom_low = np.maximum(df.loc[mask_low_has, "p5"].abs(), eps)
    df.loc[mask_low_has, "deviation_low"] = (df.loc[mask_low_has, "p5"] - df.loc[mask_low_has, "value"]) / denom_low
    df.loc[mask_low_has, "is_low_anomaly"] = df.loc[mask_low_has, "value"] <= df.loc[mask_low_has, "p5"]

    # unify anomaly flag and a single deviation numeric (positive)
    df["is_anomaly"] = df["is_high_anomaly"] | df["is_low_anomaly"]
    df["deviation_abs"] = 0.0
    # use deviation for high, deviation_low for low
    df.loc[mask_high_has, "deviation_abs"] = df.loc[mask_high_has, "deviation"].abs().fillna(0)
    df.loc[mask_low_has, "deviation_abs"] = df.loc[mask_low_has, "deviation_low"].abs().fillna(0)
    return df

trace_checks = compute_row_flags(trace_checks)

#  Save full candidates (all checked rows with stats) for next step
trace_anomaly_candidates_df = trace_checks.copy()

# 4a) For each fault_start_timestamp, select top 20 rows by deviation_abs (desc)
# Keep requested columns and rename timestamp to checked_timestamp for clarity
trace_anomaly_candidates_df["checked_timestamp"] = trace_anomaly_candidates_df["timestamp"].astype("Int64")

cols_out = ["fault_start_timestamp", "checked_timestamp", "cmdb_id", "trace_name", "value", "p95", "p5", "anomaly_type_guess", "is_anomaly", "deviation_abs"]

# Filter only actual anomalies or keep all but sort by deviation; the prompt asks to flag anomalies and return top by abs deviation.
# We'll return top rows by deviation_abs (including non-anomalies if they have deviation computed; is_anomaly marks actual ones)
top_by_start = (
    trace_anomaly_candidates_df
    .loc[:, cols_out]
    .sort_values(["fault_start_timestamp", "deviation_abs"], ascending=[True, False])
    .groupby("fault_start_timestamp", group_keys=False)
    .head(20)
    .reset_index(drop=True)
)

# 4b) For each fault_start_timestamp, count anomalous traces per cmdb_id (is_anomaly True), top 10 cmdb_ids by count
anomalies_only = trace_anomaly_candidates_df[trace_anomaly_candidates_df["is_anomaly"]].copy()
if not anomalies_only.empty:
    agg_counts = (
        anomalies_only
        .groupby(["fault_start_timestamp", "cmdb_id"], as_index=False)
        .size()
        .rename(columns={"size": "anomalous_trace_count"})
    )
    top_counts_per_start = (
        agg_counts
        .sort_values(["fault_start_timestamp", "anomalous_trace_count", "cmdb_id"], ascending=[True, False, True])
        .groupby("fault_start_timestamp", group_keys=False)
        .head(10)
        .reset_index(drop=True)
    )
else:
    top_counts_per_start = pd.DataFrame(columns=["fault_start_timestamp", "cmdb_id", "anomalous_trace_count"])

# Keep outputs compact: show shapes and head samples
trace_thresholds_df.shape, top_by_start, top_counts_per_start.head(20)

```
Out[6]:
```
summary = (
    "Trace check summary (plain English):\n\n"
    "1) Thresholds computed for traces: trace_thresholds_df contains 1,232 (cmdb_id, trace_name) entries.\n\n"
    "2) Timepoints checked: for each refined fault start we inspected the start timestamp and +60s. Primary starts were 1647840600 and 1647841020.\n\n"
    "3) Main findings:\n"
    "   - Strong high-side anomalies in latency/duration metrics (trace.*.duration_p95 and duration_mean) at/near fault starts. Examples: cartservice2-0 and frontend2-0 show very large deviations (duration values many orders of magnitude above p95).\n"
    "   - These high deviations indicate immediate propagation of latency from frontends to backend services (cartservice, checkoutservice, productcatalogservice) around the fault starts.\n"
    "   - The anomaly counts per start show the most affected components: at 1647840600, checkoutservice-2/checkoutservice-0 and frontend-0/-1/-2 are top; similarly at 1647841020, checkoutservice and frontend pods again top the list. Many cartservice and productcatalogservice entries also appear.\n\n"
    "4) Caveats: many trace p95 or p5 baseline values are extremely small or zero, which inflates computed deviations; interpret absolute magnitudes with caution and cross-check raw values.\n\n"
    "5) Recommended next steps: prioritize investigation of latency propagation paths (frontend -> cartservice -> checkoutservice -> productcatalogservice). Focus on the top affected cmdb_ids (checkoutservice-*, frontend-*) at the two start timestamps and validate whether the high duration values correspond to increased request timeouts, retries, or downstream congestion.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

((1232, 4),     fault_start_timestamp  checked_timestamp                  cmdb_id                              trace_name      value        p95        p5 anomaly_type_guess  is_anomaly  deviation_abs
0              1647840600         1647840600           cartservice2-0                 trace.self.duration_p95   0.179000   0.000000   0.00000               high        True  179000.000000
1              1647840600         1647840600           cartservice2-0     trace.from_frontend2-0.duration_p95  11.446000   0.001000   0.00100               high        True   11445.000000
2              1647840600         1647840600              frontend2-0    trace.to_cartservice2-0.duration_p95  11.446000   0.001000   0.00100               high        True   11445.000000
3              1647840600         1647840600           cartservice2-0    trace.from_frontend2-0.duration_mean   1.410911   0.000269   0.00014               high        True    5242.073207
4              1647840600         1647840600              frontend2-0   trace.to_cartservice2-0.duration_mean   1.410911   0.000269   0.00014               high        True    5242.073207
..                    ...                ...                      ...                                     ...        ...        ...       ...                ...         ...            ...
35             1647841020         1647841020  productcatalogservice-2         trace.from_frontend-2.row_count  43.000000  47.000000  13.00000                low       False       2.307692
36             1647841020         1647841020        checkoutservice-0               trace.from_root.row_count  11.000000  11.550000   3.45000                low       False       2.188406
37             1647841020         1647841020            cartservice-1  trace.from_checkoutservice-0.row_count   3.000000   3.000000   1.00000                low       False       2.000000
38             1647841020         1647841020            cartservice-2  trace.from_checkoutservice-0.row_count   3.000000   2.600000   1.00000                low       False       2.000000
39             1647841020         1647841020        checkoutservice-0        trace.to_cartservice-1.row_count   3.000000   3.000000   1.00000                low       False       2.000000

[40 rows x 10 columns],     fault_start_timestamp                  cmdb_id  anomalous_trace_count
0              1647840600        checkoutservice-2                     82
1              1647840600        checkoutservice-0                     81
2              1647840600               frontend-0                     77
3              1647840600        checkoutservice-1                     69
4              1647840600               frontend-1                     62
5              1647840600               frontend-2                     51
6              1647840600              frontend2-0                     43
7              1647840600            cartservice-0                     39
8              1647840600  productcatalogservice-2                     37
9              1647840600  productcatalogservice-1                     34
10             1647841020        checkoutservice-2                     83
11             1647841020        checkoutservice-1                     77
12             1647841020               frontend-1                     77
13             1647841020        checkoutservice-0                     74
14             1647841020               frontend-0                     60
15             1647841020               frontend-2                     46
16             1647841020            cartservice-2                     35
17             1647841020            cartservice-1                     34
18             1647841020            cartservice-0                     30
19             1647841020  productcatalogservice-0                     26)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# Reuse cached variables: refined_faults_df, trace_anomaly_candidates_df, error_df, log_df, trace_thresholds_df, top_counts_per_start (if available)

# 1) Gather distinct fault_start_timestamps programmatically
start_timestamps = sorted(refined_faults_df["fault_start_timestamp"].dropna().unique().tolist())

# Prepare containers for outputs
error_agg_rows = []
log_checks_rows = []

# 1) Error logs processing
for start in start_timestamps:
    start = int(start)
    end = start + 60
    window = error_df[(error_df["timestamp"] >= start) & (error_df["timestamp"] <= end)].copy()
    if window.empty:
        continue
    # group by cmdb_id: count and earliest timestamp
    grp = (
        window.groupby("cmdb_id", as_index=False)
        .agg(message_count=("message", "size"), earliest_message_timestamp=("timestamp", "min"))
    )
    # top 10 cmdb_ids by count
    top_cmdbs = grp.sort_values("message_count", ascending=False).head(10)
    # collect up to 3 distinct sample messages per cmdb_id (truncated to 120 chars)
    for _, row in top_cmdbs.iterrows():
        cmdb = row["cmdb_id"]
        sub = window[window["cmdb_id"] == cmdb]
        samples = (
            sub["message"].astype(str).drop_duplicates().head(3).apply(lambda s: s[:120]).tolist()
        )
        error_agg_rows.append({
            "fault_start_timestamp": start,
            "cmdb_id": cmdb,
            "message_count": int(row["message_count"]),
            "earliest_message_timestamp": int(row["earliest_message_timestamp"]),
            "sample_messages": " || ".join(samples)
        })

error_aggregate_df = pd.DataFrame(error_agg_rows)
# Keep compact: limit to 20 rows
error_aggregate_df = error_aggregate_df.sort_values(["fault_start_timestamp", "message_count"], ascending=[True, False]).head(20).reset_index(drop=True)

# 2a) Compute global P95 and P5 per (cmdb_id, log_name) across full log_df
log_thresholds_df = (
    log_df
    .groupby(["cmdb_id", "log_name"], as_index=False)["value"]
    .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
)

# Ensure top_counts_per_start exists; if not, derive from trace_anomaly_candidates_df
try:
    _ = top_counts_per_start
except NameError:
    # derive top_counts_per_start similar to prior step: group anomalies by fault_start_timestamp and cmdb_id
    if "trace_anomaly_candidates_df" in globals():
        anomalies_tmp = trace_anomaly_candidates_df[trace_anomaly_candidates_df["is_anomaly"]].copy()
        top_counts_per_start = (
            anomalies_tmp.groupby(["fault_start_timestamp", "cmdb_id"], as_index=False)
            .size()
            .rename(columns={"size": "anomalous_trace_count"})
        )
    else:
        top_counts_per_start = pd.DataFrame(columns=["fault_start_timestamp", "cmdb_id", "anomalous_trace_count"])

# Helper to classify log_name semantics: error -> high (higher is bad), row_count/count -> low (drop is bad)
def classify_log(name):
    n = str(name).lower()
    if "error" in n:
        return "high"
    if "row_count" in n or "row" in n or "count" in n:
        return "low"
    # default to high
    return "high"

# 2b & 2c) For each fault start, use top-affected cmdb_ids from traces and extract log rows at start and start+60
for start in start_timestamps:
    start = int(start)
    checked_ts = {start, start + 60}
    # find top cmdb_ids for this start (from top_counts_per_start)
    top_cms = top_counts_per_start[top_counts_per_start["fault_start_timestamp"] == start]
    if top_cms.empty:
        continue
    top_cms_list = top_cms.sort_values("anomalous_trace_count", ascending=False).head(10)["cmdb_id"].unique().tolist()
    if not top_cms_list:
        continue
    # extract log rows matching these cmdb_ids and checked timestamps
    logs_sel = log_df[(log_df["timestamp"].isin(checked_ts)) & (log_df["cmdb_id"].isin(top_cms_list))].copy()
    if logs_sel.empty:
        continue
    # join thresholds
    logs_sel = logs_sel.merge(log_thresholds_df, on=["cmdb_id", "log_name"], how="left")
    # classify and flag anomalies
    logs_sel["semantics"] = logs_sel["log_name"].apply(classify_log)
    # numeric conversion
    logs_sel["value"] = pd.to_numeric(logs_sel["value"], errors="coerce")
    logs_sel["p95"] = pd.to_numeric(logs_sel["p95"], errors="coerce")
    logs_sel["p5"] = pd.to_numeric(logs_sel["p5"], errors="coerce")
    # compute anomaly flags
    logs_sel["is_anomalous"] = False
    # high semantics: value >= p95
    mask_high = logs_sel["semantics"] == "high"
    mask_high_has = mask_high & logs_sel["p95"].notna()
    logs_sel.loc[mask_high_has, "is_anomalous"] = logs_sel.loc[mask_high_has, "value"] >= logs_sel.loc[mask_high_has, "p95"]
    # low semantics: value <= p5
    mask_low = logs_sel["semantics"] == "low"
    mask_low_has = mask_low & logs_sel["p5"].notna()
    logs_sel.loc[mask_low_has, "is_anomalous"] = logs_sel.loc[mask_low_has, "value"] <= logs_sel.loc[mask_low_has, "p5"]
    # compact row output
    logs_sel_out = logs_sel.rename(columns={"timestamp": "checked_timestamp"})[
        ["checked_timestamp", "cmdb_id", "log_name", "value", "p95", "p5", "is_anomalous"]
    ].copy()
    # append to results
    log_checks_rows.append(logs_sel_out)

# Concatenate log checks (may be multiple DataFrames)
if log_checks_rows:
    logs_extracted_df = pd.concat(log_checks_rows, ignore_index=True)
    # keep compact: limit to 20 rows sorted by checked_timestamp then anomalous first
    logs_extracted_df = logs_extracted_df.sort_values(["checked_timestamp", "is_anomalous"], ascending=[True, False]).head(20).reset_index(drop=True)
else:
    logs_extracted_df = pd.DataFrame(columns=["checked_timestamp", "cmdb_id", "log_name", "value", "p95", "p5", "is_anomalous"])

# Save outputs in kernel: log_thresholds_df, error_aggregate_df, logs_extracted_df
log_thresholds_df.shape, error_aggregate_df, logs_extracted_df

```
Out[7]:
```
Summary of findings for each fault start (compact, plain English):

Error logs (within [start, start+60]):
- Main sources of error log messages are the frontend pods. Top counts:
  - At 1647840600: frontend-1 (16 messages), frontend-0 (13), frontend-2 (6). Sample messages: "severity: warning, message: failed to retrieve..." (truncated).
  - At 1647841020: frontend-1 (16), frontend-0 (10), frontend-2 (7). Same sample messages.
- Interpretation: frontends emitted repeated "failed to retrieve" warnings immediately around both fault starts — these are the most prominent error-log signals.

Log metrics (global thresholds computed; log_thresholds_df saved):
- Many log.error_count rows show value=0 with p95=0 → flagged as anomalous by the simple rule (value >= p95). These are noisy/false-positives because the baseline p95 is zero.
- More meaningful anomalies (low-side) detected:
  - At 1647840600, productcatalogservice-2.log.row_count = 627 which is below its global P5 (628.35) and below P95 (765.75) — flagged as anomalous (drop in traffic).
- Other components (checkoutservice, cartservice, frontend, productcatalogservice) appear in the checked set; many row_count values are below their P95 but not always below P5, so only some are flagged as true low anomalies.
- A few log.error_count entries at later checked timestamps (e.g., 1647840660) also appear flagged (again often because of p95==0).

Overall interpretation and recommended next steps:
- Error logs point strongly to frontend pods failing to "retrieve" immediately at both fault starts. Investigate frontend error messages and upstream calls they make.
- Trace and log evidence shows downstream impact on productcatalogservice and checkoutservice (reduced row_count / high trace durations). Focus initial triage on frontend → cartservice → checkoutservice → productcatalogservice paths.
- Caveat: many alerts are caused by zero-baseline thresholds (p95==0) and create noisy positives (especially for log.error_count). Recompute/adjust thresholds or apply secondary rules to avoid treating zero baselines as meaningful breaches.
- Saved artifacts for follow-up: log_thresholds_df, error_aggregate_df, and the extracted logs (logs_extracted_df) in the kernel.

The original code execution output of IPython Kernel is also provided below for reference:

((62, 4),    fault_start_timestamp     cmdb_id  message_count  earliest_message_timestamp                                    sample_messages
0             1647840600  frontend-1             16                  1647840602  severity: warning, message: failed to retrieve...
1             1647840600  frontend-0             13                  1647840601  severity: warning, message: failed to retrieve...
2             1647840600  frontend-2              6                  1647840601  severity: warning, message: failed to retrieve...
3             1647841020  frontend-1             16                  1647841020  severity: warning, message: failed to retrieve...
4             1647841020  frontend-0             10                  1647841022  severity: warning, message: failed to retrieve...
5             1647841020  frontend-2              7                  1647841022  severity: warning, message: failed to retrieve...,     checked_timestamp                  cmdb_id         log_name   value      p95       p5  is_anomalous
0          1647840600            cartservice-0  log.error_count     0.0     0.00     0.00          True
1          1647840600        checkoutservice-0  log.error_count     0.0     0.00     0.00          True
2          1647840600        checkoutservice-1  log.error_count     0.0     0.00     0.00          True
3          1647840600        checkoutservice-2  log.error_count     0.0     0.00     0.00          True
4          1647840600  productcatalogservice-1  log.error_count     0.0     0.00     0.00          True
5          1647840600  productcatalogservice-2  log.error_count     0.0     0.00     0.00          True
6          1647840600  productcatalogservice-2    log.row_count   627.0   765.75   628.35          True
7          1647840600            cartservice-0    log.row_count   540.0   619.55   514.70         False
8          1647840600        checkoutservice-0    log.row_count    33.0    52.10    24.95         False
9          1647840600        checkoutservice-1    log.row_count    51.0    67.55    17.00         False
10         1647840600        checkoutservice-2    log.row_count    68.0    58.15    28.90         False
11         1647840600               frontend-0  log.error_count    13.0    16.00    11.00         False
12         1647840600               frontend-0    log.row_count   983.0  1262.65   946.50         False
13         1647840600               frontend-1  log.error_count    15.0    22.00    16.00         False
14         1647840600               frontend-1    log.row_count  1303.0  1634.10  1297.45         False
15         1647840600               frontend-2  log.error_count     6.0     7.00     0.00         False
16         1647840600               frontend-2    log.row_count   422.0   453.90     2.00         False
17         1647840600  productcatalogservice-1    log.row_count   653.0   766.15   637.05         False
18         1647840660            cartservice-0  log.error_count     0.0     0.00     0.00          True
19         1647840660        checkoutservice-0  log.error_count     0.0     0.00     0.00          True)```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

# Reuse cached variables: metric_df, thresholds_df (previous p95-only), refined_faults_df, candidates_metrics_df

# 1) Identify distinct refined fault_start_timestamps programmatically
start_timestamps = sorted(refined_faults_df["fault_start_timestamp"].dropna().unique().tolist())
# small DataFrame view for return
start_ts_df = pd.DataFrame({"fault_start_timestamp": [int(x) for x in start_timestamps]})

# 2) Compute global P95 and P5 per (cmdb_id, kpi_name) across the entire metric_df (global thresholds)
metric_thresholds_df = (
    metric_df
    .groupby(["cmdb_id", "kpi_name"], as_index=False)["value"]
    .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
)

# Helper to classify whether a KPI is likely "low-is-bad" (traffic/drop) or "high-is-bad" (resource spike)
low_like_tokens = ["row_count", "row.count", "row.counts", "count", "throughput", "recv", "receive", "tx", "transmit", "error_rate", "error.rate"]
def is_low_metric(kpi):
    k = str(kpi).lower()
    return any(tok in k for tok in low_like_tokens)

# 2a) Node-level checks for node-5 at each start timestamp
node_checks = []
for start in start_timestamps:
    ts = int(start)
    sel = metric_df[(metric_df["cmdb_id"] == "node-5") & (metric_df["timestamp"] == ts)].copy()
    if sel.empty:
        continue
    # join global thresholds
    sel = sel.merge(metric_thresholds_df, on=["cmdb_id", "kpi_name"], how="left")
    # compute deviation using p95 where available; for sorting use abs(value - p95), fallback to abs(value - p5) if p95 missing
    sel["p95"] = pd.to_numeric(sel["p95"], errors="coerce")
    sel["p5"] = pd.to_numeric(sel["p5"], errors="coerce")
    sel["value"] = pd.to_numeric(sel["value"], errors="coerce")
    # determine anomaly flag: high (value >= p95) or low (value <= p5) based on heuristic
    sel["semantics_low"] = sel["kpi_name"].apply(is_low_metric)
    sel["is_anomaly"] = False
    # high-case
    mask_high = (~sel["semantics_low"]) & sel["p95"].notna()
    sel.loc[mask_high, "is_anomaly"] = sel.loc[mask_high, "value"] >= sel.loc[mask_high, "p95"]
    # low-case
    mask_low = (sel["semantics_low"]) & sel["p5"].notna()
    sel.loc[mask_low, "is_anomaly"] = sel.loc[mask_low, "value"] <= sel.loc[mask_low, "p5"]
    # deviation for sorting: prefer |value - p95|, else |value - p5|, else |value|
    sel["dev_p95"] = (sel["value"] - sel["p95"]).abs()
    sel["dev_p5"] = (sel["value"] - sel["p5"]).abs()
    sel["deviation_sort"] = sel["dev_p95"].fillna(sel["dev_p5"]).fillna(sel["value"].abs())
    # attach checked_timestamp column
    sel["checked_timestamp"] = sel["timestamp"].astype("Int64")
    # keep requested columns
    sel_out = sel[["checked_timestamp", "cmdb_id", "kpi_name", "value", "p95", "p5", "is_anomaly", "deviation_sort"]].copy()
    sel_out["fault_start_timestamp"] = int(ts)
    node_checks.append(sel_out)

# concat and pick top 20 rows sorted by deviation_sort desc (per instruction overall)
if node_checks:
    node5_checked_df = pd.concat(node_checks, ignore_index=True)
    node5_checked_df = node5_checked_df.sort_values(by=["fault_start_timestamp", "deviation_sort"], ascending=[True, False])
    node5_checked_df = node5_checked_df.head(20).reset_index(drop=True)
else:
    node5_checked_df = pd.DataFrame(columns=["fault_start_timestamp","checked_timestamp","cmdb_id","kpi_name","value","p95","p5","is_anomaly","deviation_sort"])

# 2b) Cartservice pods checks for specified kpis at each fault timestamp
cart_cmdbs = ["cartservice-0", "cartservice-1", "cartservice-2"]
kpi_tokens = ["container_cpu_cfs_throttled", "container_cpu_cfs_periods"]

cart_checks = []
for start in start_timestamps:
    ts = int(start)
    sel = metric_df[
        (metric_df["cmdb_id"].isin(cart_cmdbs)) &
        (metric_df["timestamp"] == ts) &
        (metric_df["kpi_name"].str.contains("|".join(kpi_tokens), case=False, regex=True))
    ].copy()
    if sel.empty:
        continue
    sel = sel.merge(metric_thresholds_df, on=["cmdb_id", "kpi_name"], how="left")
    sel["p95"] = pd.to_numeric(sel["p95"], errors="coerce")
    sel["value"] = pd.to_numeric(sel["value"], errors="coerce")
    # anomaly: value >= p95 (as requested)
    sel["is_anomaly"] = False
    mask_has = sel["p95"].notna()
    sel.loc[mask_has, "is_anomaly"] = sel.loc[mask_has, "value"] >= sel.loc[mask_has, "p95"]
    sel["deviation_sort"] = (sel["value"] - sel["p95"]).abs().fillna(sel["value"].abs())
    sel["checked_timestamp"] = sel["timestamp"].astype("Int64")
    sel_out = sel[["checked_timestamp", "cmdb_id", "kpi_name", "value", "p95", "is_anomaly", "deviation_sort"]].copy()
    sel_out["fault_start_timestamp"] = int(ts)
    cart_checks.append(sel_out)

if cart_checks:
    cartservice_checked_df = pd.concat(cart_checks, ignore_index=True)
    cartservice_checked_df = cartservice_checked_df.sort_values(by=["fault_start_timestamp", "deviation_sort"], ascending=[True, False])
    # show up to 20 rows (the user requested show all results up to 20)
    cartservice_checked_df = cartservice_checked_df.head(20).reset_index(drop=True)
else:
    cartservice_checked_df = pd.DataFrame(columns=["fault_start_timestamp","checked_timestamp","cmdb_id","kpi_name","value","p95","is_anomaly","deviation_sort"])

# Save outputs in kernel: node5_checked_df, cartservice_checked_df
# Return compact outputs: start_ts_df, node5_checked_df (top 20), cartservice_checked_df (up to 20)
start_ts_df, node5_checked_df, cartservice_checked_df

```
Out[8]:
```
Summary (plain English):

1) Fault start timestamps found:
- 1647840600 and 1647841020.

2) Node-5 (node-level) check at each fault timestamp:
- Reviewed top node-5 KPIs at the fault timestamps. Most node metrics are near their baselines; notable points:
  - system.cpu.pct_usage ~6.32% vs p95 ~6.89% — not above p95 (no node CPU spike).
  - system.io.await and w_await are zero (no I/O stall signal).
  - system.disk.free and system.mem.free have tiny deviations around thresholds (one-off small differences), but these are minor.
- Conclusion: no clear node-level CPU contention or sustained node resource spike on node-5 at the checked timestamps.

3) Cartservice pods (container-level) check at each fault timestamp:
- Checked container_cpu_cfs_periods and container_cpu_cfs_throttled for cartservice-0/1/2.
- At 1647840600: container_cpu_cfs_periods values are below their p95 (no spike). Many container_cpu_cfs_throttled metrics are zero; some are flagged only because p95==0 (noisy/false positives).
- At 1647841020: cartservice-1 shows a genuine throttling signal: container_cpu_cfs_throttled value = 1.0 with p95 = 0.55 (is_anomaly = True). This indicates actual container CPU throttling on cartservice-1 at 1647841020.
- Many other cartservice throttled metrics are 0 with p95 == 0 and are therefore unreliable signals (they create false positives).

4) Overall interpretation and next actions:
- The evidence points to container-level CPU throttling affecting cartservice pods, not a node-wide CPU overload. The strongest, non-spurious throttling hit is cartservice-1 at 1647841020.
- Recommend investigating cartservice-1 container CPU limits / cgroup throttling (e.g., CPU requests/limits, throttling counters, scheduler behavior), and check whether other cartservice pods experienced similar throttling or were impacted by the same root cause.
- Also advise filtering out/adjusting zero-baseline thresholds (p95 == 0) to avoid noisy false positives when prioritizing investigation.

The original code execution output of IPython Kernel is also provided below for reference:

(   fault_start_timestamp
0             1647840600
1             1647841020,     checked_timestamp cmdb_id                      kpi_name         value           p95            p5  is_anomaly  deviation_sort  fault_start_timestamp
0          1647840600  node-5              system.disk.used  4.481210e+07  4.492522e+07  4.481274e+07       False     113119.6000             1647840600
1          1647840600  node-5              system.disk.free  5.489677e+08  5.489671e+08  5.488546e+08        True        633.6000             1647840600
2          1647840600  node-5         system.net.bytes_rcvd  2.446029e+04  2.507341e+04  2.317657e+04       False        613.1225             1647840600
3          1647840600  node-5         system.net.bytes_sent  3.849264e+04  3.893685e+04  3.733352e+04       False        444.2090             1647840600
4          1647840600  node-5                 system.io.w_s  0.000000e+00  8.405000e+01  0.000000e+00       False         84.0500             1647840600
5          1647840600  node-5      system.os.nofile.current  1.404800e+04  1.411200e+04  1.401600e+04       False         64.0000             1647840600
6          1647840600  node-5               system.mem.used  8.265000e+03  8.285100e+03  8.265000e+03       False         20.1000             1647840600
7          1647840600  node-5   system.net.udp.in_datagrams  6.175000e+01  7.465450e+01  6.220000e+01       False         12.9045             1647840600
8          1647840600  node-5  system.net.udp.out_datagrams  6.175000e+01  7.465450e+01  6.220000e+01       False         12.9045             1647840600
9          1647840600  node-5       system.net.tcp.out_segs  1.966100e+02  2.056415e+02  1.929465e+02       False          9.0315             1647840600
10         1647840600  node-5               system.mem.free  1.858300e+04  1.857580e+04  1.845890e+04        True          7.2000             1647840600
11         1647840600  node-5        system.net.tcp.in_segs  1.611000e+02  1.681810e+02  1.598625e+02       False          7.0810             1647840600
12         1647840600  node-5               system.io.await  0.000000e+00  2.692500e+00  0.000000e+00       False          2.6925             1647840600
13         1647840600  node-5             system.io.w_await  0.000000e+00  2.692500e+00  0.000000e+00       False          2.6925             1647840600
14         1647840600  node-5         system.fs.inodes.used  3.233570e+05  3.233590e+05  3.233570e+05       False          2.0000             1647840600
15         1647840600  node-5  system.net.packets_out.count  3.486000e+01  3.666450e+01  3.387400e+01       False          1.8045             1647840600
16         1647840600  node-5   system.net.packets_in.count  2.580000e+01  2.752750e+01  2.492800e+01       False          1.7275             1647840600
17         1647840600  node-5                system.io.util  0.000000e+00  7.150000e-01  0.000000e+00       False          0.7150             1647840600
18         1647840600  node-5   system.net.tcp.retrans_segs  2.000000e-02  7.250000e-01  0.000000e+00       False          0.7050             1647840600
19         1647840600  node-5          system.cpu.pct_usage  6.320000e+00  6.889500e+00  5.929000e+00       False          0.5695             1647840600,     checked_timestamp        cmdb_id                                           kpi_name       value         p95  is_anomaly  deviation_sort  fault_start_timestamp
0          1647840600  cartservice-2         container.node-5.container_cpu_cfs_periods  159.500000  190.050000       False       30.550000             1647840600
1          1647840600  cartservice-1         container.node-5.container_cpu_cfs_periods  180.500000  210.275000       False       29.775000             1647840600
2          1647840600  cartservice-0         container.node-5.container_cpu_cfs_periods  199.500000  201.275000       False        1.775000             1647840600
3          1647840600  cartservice-1  container.node-5.container_cpu_cfs_throttled_p...    0.000000    0.550000       False        0.550000             1647840600
4          1647840600  cartservice-1  container.node-5.container_cpu_cfs_throttled_s...    0.000000    0.013367       False        0.013367             1647840600
5          1647840600  cartservice-0  container.node-5.container_cpu_cfs_throttled_p...    0.000000    0.000000        True        0.000000             1647840600
6          1647840600  cartservice-0  container.node-5.container_cpu_cfs_throttled_s...    0.000000    0.000000        True        0.000000             1647840600
7          1647840600  cartservice-2  container.node-5.container_cpu_cfs_throttled_p...    0.000000    0.000000        True        0.000000             1647840600
8          1647840600  cartservice-2  container.node-5.container_cpu_cfs_throttled_s...    0.000000    0.000000        True        0.000000             1647840600
9          1647841020  cartservice-1         container.node-5.container_cpu_cfs_periods  154.500000  210.275000       False       55.775000             1647841020
10         1647841020  cartservice-2         container.node-5.container_cpu_cfs_periods  145.000000  190.050000       False       45.050000             1647841020
11         1647841020  cartservice-0         container.node-5.container_cpu_cfs_periods  167.500000  201.275000       False       33.775000             1647841020
12         1647841020  cartservice-1  container.node-5.container_cpu_cfs_throttled_p...    1.000000    0.550000        True        0.450000             1647841020
13         1647841020  cartservice-1  container.node-5.container_cpu_cfs_throttled_s...    0.024304    0.013367        True        0.010937             1647841020
14         1647841020  cartservice-0  container.node-5.container_cpu_cfs_throttled_p...    0.000000    0.000000        True        0.000000             1647841020
15         1647841020  cartservice-0  container.node-5.container_cpu_cfs_throttled_s...    0.000000    0.000000        True        0.000000             1647841020
16         1647841020  cartservice-2  container.node-5.container_cpu_cfs_throttled_p...    0.000000    0.000000        True        0.000000             1647841020
17         1647841020  cartservice-2  container.node-5.container_cpu_cfs_throttled_s...    0.000000    0.000000        True        0.000000             1647841020)```
```